# Transformer for Turbofan Engine RUL Prediction

## Overview

This educational notebook demonstrates how to use a **Transformer** model for time series prediction of Remaining Useful Life (RUL) of turbofan engines.

## Learning Objectives

By the end of this notebook, you will:
- Understand the Transformer architecture and attention mechanism
- Learn how self-attention works for sequence modeling
- Build a Transformer model for time series prediction
- Compare Transformer performance with RNN/LSTM/TCN approaches

## What is a Transformer?

**Transformers** are a revolutionary architecture that uses **attention mechanisms** instead of recurrence or convolution. Originally developed for NLP, they've shown great success in time series tasks.

### Key Innovation: Self-Attention

**Self-Attention** allows the model to:
- Look at all positions in the sequence simultaneously
- Learn which timesteps are most important for prediction
- Capture relationships between any two timesteps (not just adjacent ones)
- Weight different parts of the sequence dynamically

### Architecture Components:

1. **Multi-Head Attention**: 
   - Multiple attention mechanisms in parallel
   - Each "head" learns different relationships
   - Combines information from all heads

2. **Feed-Forward Network**: 
   - Processes attention outputs
   - Adds non-linearity

3. **Layer Normalization**: 
   - Stabilizes training
   - Applied before and after attention

4. **Residual Connections**: 
   - Helps with gradient flow
   - Enables training deep networks

### Advantages:
- **Parallel processing**: Can process entire sequence at once
- **Long-range dependencies**: Attention can connect any two timesteps
- **Interpretability**: Attention weights show what the model focuses on
- **No recurrence**: Faster training than RNNs

### When to Use:
- When you need to model complex relationships across the entire sequence
- When interpretability of attention patterns is valuable
- When you have sufficient data (Transformers can be data-hungry)
- When you want state-of-the-art performance


## 1. Import Libraries

### Purpose
Import TensorFlow/Keras components for building Transformer architecture, including MultiHeadAttention and LayerNormalization layers.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time  # For measuring training time
warnings.filterwarnings('ignore')

# Configure TensorFlow to use CPU only (avoids CUDA/libdevice issues)
# IMPORTANT: Set this BEFORE importing TensorFlow for it to take effect
# If you still see GPU errors, restart the kernel and run this cell first!
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Disable GPU, use CPU only
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings

# Deep Learning
import tensorflow as tf

# Aggressively disable GPU and force CPU usage
try:
    # Hide all GPU devices
    tf.config.set_visible_devices([], 'GPU')
    # Set memory growth to prevent GPU allocation
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
except:
    pass

# Force CPU device placement for all operations
tf.config.set_soft_device_placement(True)
with tf.device('/CPU:0'):
    # This ensures CPU is used by default
    pass

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, Input, LayerNormalization, 
                                     MultiHeadAttention, GlobalAveragePooling1D, 
                                     Embedding, Add)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Verify GPU is disabled
print(f"\n📊 TensorFlow Device Configuration:")
print(f"   Available GPUs: {len(tf.config.list_physical_devices('GPU'))}")
print(f"   Available CPUs: {len(tf.config.list_physical_devices('CPU'))}")
if len(tf.config.list_physical_devices('GPU')) == 0:
    print("   ✅ GPU successfully disabled - using CPU only")
else:
    print("   ⚠️  WARNING: GPU still detected! Please restart kernel and run this cell first.")


## 2. Load and Prepare Data

### Purpose
Load the dataset. Transformers process sequences differently than RNNs - they use attention to relate all timesteps simultaneously.


In [ ]:
# Define data path
data_path = Path('../../dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
           'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data"""
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    # CSV files already have headers, so no need for sep, header=None, or names
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Number of engines in training: {train_df['unit'].nunique()}")
print(f"Number of engines in test: {test_df['unit'].nunique()}")


## 3. Create Time Series Sequences

### Purpose
Create sequences for Transformer processing. Transformers can attend to all positions in the sequence simultaneously, making them powerful for capturing complex temporal relationships.


In [ ]:
def create_sequences(data, sequence_length=30):
    """Create sequences for time series prediction"""
    sequences = []
    targets = []
    
    # Select features (using actual field names)
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

# For test data
def create_test_sequences(data, sequence_length=30):
    """Create test sequences"""
    sequences = []
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        if len(unit_features) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            padding = np.zeros((sequence_length - len(unit_features), len(feature_cols)))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

X_test_seq = create_test_sequences(test_df, sequence_length)
y_test = rul_df['RUL'].values

print(f"Training sequences shape: {X_train_seq.shape}")
print(f"Test sequences shape: {X_test_seq.shape}")


## 4. Data Normalization

### Purpose
Normalize data for Transformer training. Attention mechanisms work best with normalized inputs to ensure stable gradient flow.


In [ ]:
# Normalize features
feature_scaler = MinMaxScaler()
n_samples, n_timesteps, n_features = X_train_seq.shape
X_train_reshaped = X_train_seq.reshape(-1, n_features)
X_train_scaled = feature_scaler.fit_transform(X_train_reshaped)
X_train_seq_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

# Normalize test data
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = feature_scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape)

# Normalize targets
target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train_seq.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

print("Data normalized successfully!")


## 5. Build Transformer Model

### Purpose
Construct a Transformer architecture with self-attention for time series prediction.

### Architecture Components:

1. **Transformer Encoder Block**:
   - **Multi-Head Self-Attention**: 
     - Computes attention between all timesteps
     - Multiple "heads" learn different types of relationships
     - Allows model to focus on important timesteps
   - **Feed-Forward Network**: 
     - Processes attention outputs
     - Adds non-linear transformations
   - **Layer Normalization**: 
     - Applied before and after attention
     - Stabilizes training
   - **Residual Connections**: 
     - Adds input to output
     - Helps with gradient flow

2. **Stacked Encoders**:
   - Multiple encoder blocks in sequence
   - Each block refines the representation
   - Deeper = more complex pattern learning

3. **Global Pooling**:
   - Aggregates sequence representation
   - Converts variable-length sequence to fixed-size vector

4. **MLP Head**:
   - Final layers that map to RUL prediction
   - Fully connected layers

### How Attention Works:
- For each timestep, computes attention scores with all other timesteps
- Higher scores = more important relationships
- Model learns which timesteps to focus on for RUL prediction
- Can capture both short-term and long-term dependencies simultaneously


In [ ]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    """Transformer encoder block"""
    # Multi-head self-attention
    attention_output = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    attention_output = Dropout(dropout)(attention_output)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
    # Feed-forward network
    ffn_output = Dense(ff_dim, activation="relu")(out1)
    ffn_output = Dense(inputs.shape[-1])(ffn_output)
    ffn_output = Dropout(dropout)(ffn_output)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output)
    
    return out2

def build_transformer_model(sequence_length, n_features, head_size=64, num_heads=4, 
                           ff_dim=128, num_transformer_blocks=2, mlp_units=[128, 64], 
                           dropout=0.1, mlp_dropout=0.3):
    """Build Transformer model for time series prediction"""
    # Force CPU device placement to avoid GPU errors
    with tf.device('/CPU:0'):
        inputs = Input(shape=(sequence_length, n_features))
        
        # Project input to model dimension
        x = Dense(head_size * num_heads)(inputs)
        
        # Stack transformer blocks
        for _ in range(num_transformer_blocks):
            x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
        
        # Global pooling
        x = GlobalAveragePooling1D()(x)
        
        # MLP head
        for dim in mlp_units:
            x = Dense(dim, activation="relu")(x)
            x = Dropout(mlp_dropout)(x)
        
        outputs = Dense(1)(x)
        
        model = Model(inputs, outputs)
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build model
n_features = X_train_seq_scaled.shape[2]
model = build_transformer_model(sequence_length, n_features)
model.summary()


## 6. Train Model

### Purpose
Train the Transformer model. Transformers:
- Can train faster than RNNs (parallel processing)
- May require more data to reach optimal performance
- Benefit from careful learning rate scheduling


In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train model
history = model.fit(
    X_train_seq_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


## 7. Evaluate Model

### Purpose
Evaluate Transformer performance. Compare with other models to see if attention mechanism provides advantages for this task.


In [ ]:
# Predictions
y_train_pred_scaled = model.predict(X_train_seq_scaled, verbose=0)
y_test_pred_scaled = model.predict(X_test_seq_scaled, verbose=0)

# Inverse transform
y_train_pred = target_scaler.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred = target_scaler.inverse_transform(y_test_pred_scaled).flatten()

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train_seq, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train_seq, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train_seq, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("Transformer Model Performance")
print("="*60)
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")


## 8. Visualizations

### Purpose
Visualize Transformer training and predictions. The attention mechanism (though not visualized here) allows the model to learn which timesteps are most important for RUL prediction.


In [ ]:
# Import additional libraries for cross-validation
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, r2_score

print("✅ Cross-validation libraries imported!")

In [ ]:
def evaluate_transformer_hyperparameters(X, y, param_combinations, n_splits=3, epochs=20, verbose=0):
    """Evaluate Transformer hyperparameters with cross-validation"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    
    print(f"🔍 Evaluating {len(param_combinations)} Transformer parameter combinations...")
    print(f"   Using {n_splits}-fold cross-validation")
    print("=" * 70)
    
    for idx, params in enumerate(param_combinations, 1):
        print(f"\n[{idx}/{len(param_combinations)}] Testing: {params}")
        fold_scores = []
        fold_r2_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
            if verbose > 0:
                print(f"  Fold {fold}/{n_splits}...", end=" ")
            
            X_train_fold = X[train_idx]
            y_train_fold = y[train_idx]
            X_val_fold = X[val_idx]
            y_val_fold = y[val_idx]
            
            with tf.device('/CPU:0'):
                inputs = Input(shape=(X.shape[1], X.shape[2]))
                x = Dense(params['head_size'] * params['num_heads'])(inputs)
                
                for _ in range(params['num_blocks']):
                    x = transformer_encoder(x, params['head_size'], params['num_heads'], 
                                           params['ff_dim'], params['dropout'])
                
                x = GlobalAveragePooling1D()(x)
                for dim in params['mlp_units']:
                    x = Dense(dim, activation="relu")(x)
                    x = Dropout(params['dropout'])(x)
                
                outputs = Dense(1)(x)
                model = Model(inputs, outputs)
                model.compile(optimizer=Adam(learning_rate=params['learning_rate']), 
                            loss='mse', metrics=['mae'])
            
            model.fit(X_train_fold, y_train_fold, epochs=epochs, batch_size=params['batch_size'],
                     validation_data=(X_val_fold, y_val_fold), verbose=0,
                     callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])
            
            y_pred_fold = model.predict(X_val_fold, verbose=0)
            rmse_fold = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
            r2_fold = r2_score(y_val_fold, y_pred_fold)
            
            fold_scores.append(rmse_fold)
            fold_r2_scores.append(r2_fold)
            
            if verbose > 0:
                print(f"RMSE: {rmse_fold:.4f}, R²: {r2_fold:.4f}")
        
        avg_rmse = np.mean(fold_scores)
        std_rmse = np.std(fold_scores)
        avg_r2 = np.mean(fold_r2_scores)
        std_r2 = np.std(fold_r2_scores)
        
        results.append({
            'params': params,
            'avg_rmse': avg_rmse,
            'std_rmse': std_rmse,
            'avg_r2': avg_r2,
            'std_r2': std_r2
        })
        
        print(f"  ✅ Average RMSE: {avg_rmse:.4f} (±{std_rmse:.4f})")
        print(f"     Average R²: {avg_r2:.4f} (±{std_r2:.4f})")
    
    return results

print("✅ Transformer hyperparameter evaluation function created!")

In [ ]:
# Define Transformer hyperparameter combinations
param_combinations_transformer = [
    {'head_size': 32, 'num_heads': 2, 'ff_dim': 64, 'num_blocks': 1, 'mlp_units': [64], 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.1},
    {'head_size': 64, 'num_heads': 4, 'ff_dim': 128, 'num_blocks': 2, 'mlp_units': [128, 64], 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.1},
    {'head_size': 128, 'num_heads': 4, 'ff_dim': 256, 'num_blocks': 2, 'mlp_units': [128, 64], 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.1},
    {'head_size': 64, 'num_heads': 8, 'ff_dim': 128, 'num_blocks': 2, 'mlp_units': [128, 64], 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.1},
    {'head_size': 64, 'num_heads': 4, 'ff_dim': 128, 'num_blocks': 3, 'mlp_units': [128, 64], 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.1},
    {'head_size': 64, 'num_heads': 4, 'ff_dim': 128, 'num_blocks': 2, 'mlp_units': [128, 64], 'learning_rate': 0.0001, 'batch_size': 32, 'dropout': 0.1},
]

print(f"📋 Defined {len(param_combinations_transformer)} Transformer hyperparameter combinations")

### Run Cross-Validation for Transformer

**Note**: This will take time. Using 3-fold CV with fewer epochs for speed.

In [ ]:
# Run Transformer hyperparameter tuning
print("🚀 Starting Transformer Hyperparameter Tuning...")
print("⚠️  This may take 20-30 minutes...")
print("=" * 70)

start_time = time.time()
tuning_results_transformer = evaluate_transformer_hyperparameters(
    X_train_seq_scaled, y_train_scaled,
    param_combinations_transformer, n_splits=3, epochs=20, verbose=1
)
tuning_time = time.time() - start_time

print(f"\n✅ Transformer tuning completed in {tuning_time/60:.1f} minutes!")

# Find best
best_result_transformer = max(tuning_results_transformer, key=lambda x: x['avg_r2'])
best_params_transformer = best_result_transformer['params']

print(f"\n🏆 Best Transformer Parameters:")
print(f"   Head Size: {best_params_transformer['head_size']}, Heads: {best_params_transformer['num_heads']}")
print(f"   FF Dim: {best_params_transformer['ff_dim']}, Blocks: {best_params_transformer['num_blocks']}")
print(f"   Learning Rate: {best_params_transformer['learning_rate']}, Dropout: {best_params_transformer['dropout']}")
print(f"   Best CV R²: {best_result_transformer['avg_r2']:.4f}")

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transformer Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Model MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transformer Predictions vs Actual', fontsize=14, fontweight='bold')

# Training set
axes[0].scatter(y_train_seq, y_train_pred, alpha=0.5, s=20)
min_val = min(min(y_train_seq), min(y_train_pred))
max_val = max(max(y_train_seq), max(y_train_pred))
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL', fontsize=11)
axes[0].set_ylabel('Predicted RUL', fontsize=11)
axes[0].set_title(f'Train Set (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=20)
min_val = min(min(y_test), min(y_test_pred))
max_val = max(max(y_test), max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual RUL', fontsize=11)
axes[1].set_ylabel('Predicted RUL', fontsize=11)
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
